# 🍕 Парсер Меню Ресторана - Полностью Бесплатная Версия

**Используемые бесплатные сервисы:**
- 🔍 **Tesseract OCR** - бесплатное распознавание текста (open-source)
- 🤖 **Groq API** - бесплатный инферес LLM (8000+ токенов в мин)
- 💾 **Google Drive** - облачное хранилище (15 ГБ бесплатно)

**Ваше заведение:** Coffee Story (ID: 70000001048209528)

## 📋 БЛОК 1: Установка зависимостей

In [ ]:
# Установка Tesseract (бесплатный OCR)
!apt-get update -qq
!apt-get install -y tesseract-ocr libtesseract-dev -qq

# Python библиотеки
!pip install pytesseract pillow requests opencv-python groq -q

print("✓ Все зависимости установлены")

## 📋 БЛОК 2: Импорты и конфигурация

In [ ]:
from google.colab import drive, auth
import os
import json
import csv
from pathlib import Path
from datetime import datetime
import requests
from PIL import Image
from io import BytesIO
import pytesseract
import cv2
import numpy as np
from groq import Groq

print("✓ Все импорты успешны")

## ⚙️ БЛОК 3: КОНФИГУРАЦИЯ - ОТРЕДАКТИРУЙТЕ ЗДЕСЬ

### Получите бесплатный Groq API ключ (2 минуты):
1. Перейти: https://console.groq.com/
2. Зарегистрироваться (email или Google)
3. API Keys → Create API Key
4. Скопировать ключ сюда ↓

In [ ]:
# ===== ОТРЕДАКТИРУЙТЕ ЭТИ ПЕРЕМЕННЫЕ =====

# Groq API ключ (БЕСПЛАТНЫЙ от https://console.groq.com/)
GROQ_API_KEY = "gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# Данные заведения
FIRM_ID = "70000001048209528"  # Coffee Story
FIRM_NAME = "Coffee Story"

# Папки на Google Drive
GOOGLE_DRIVE_FOLDER = "/content/drive/MyDrive/Menu_Parser"
PHOTOS_FOLDER = "photos"
RESULTS_FOLDER = "results"
PROCESSED_FOLDER = "processed"

# ========================================

print("⚙️  Конфигурация загружена")
print(f"Firm ID: {FIRM_ID}")
print(f"Firm Name: {FIRM_NAME}")

## 📋 БЛОК 4: Подключение Google Drive и инициализация

In [ ]:
def setup_drive_folders():
    """Подключает Google Drive и создает необходимые папки"""
    
    # Подключаемся к Google Drive
    drive.mount('/content/drive', force_remount=True)
    
    base_path = Path(GOOGLE_DRIVE_FOLDER)
    base_path.mkdir(parents=True, exist_ok=True)
    
    # Создаем подпапки
    folders = {
        'photos': base_path / PHOTOS_FOLDER,
        'results': base_path / RESULTS_FOLDER,
        'processed': base_path / PROCESSED_FOLDER
    }
    
    for folder_path in folders.values():
        folder_path.mkdir(parents=True, exist_ok=True)
    
    print("✓ Google Drive подключен")
    print(f"✓ Рабочая папка: {base_path}")
    print(f"✓ Структура папок создана\n")
    
    return folders

folders = setup_drive_folders()

## 📋 БЛОК 5: Загрузка изображений

In [ ]:
def download_image(image_url, save_path):
    """Загружает изображение с URL и сохраняет локально"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(image_url, headers=headers, timeout=10)
        response.raise_for_status()
        
        img = Image.open(BytesIO(response.content))
        img.save(save_path)
        print(f"✓ Фото сохранено: {save_path.name}")
        return True
    except Exception as e:
        print(f"✗ Ошибка загрузки: {e}")
        return False

print("✓ Функция download_image готова")

## 📋 БЛОК 6: OCR - Распознавание текста (БЕСПЛАТНО через Tesseract)

In [ ]:
def preprocess_image(image_path):
    """Предобработка изображения для улучшения OCR"""
    
    img = cv2.imread(str(image_path))
    
    # Конвертировать в серый
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Увеличить контраст
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    contrast = clahe.apply(gray)
    
    # Применить пороговое значение (черный текст на белом фоне)
    _, threshold = cv2.threshold(contrast, 150, 255, cv2.THRESH_BINARY)
    
    # Убрать шум
    denoised = cv2.fastNlMeansDenoising(threshold, h=10)
    
    # Сохранить обработанное изображение
    processed_path = str(image_path).replace('.jpg', '_processed.jpg').replace('.png', '_processed.png')
    cv2.imwrite(processed_path, denoised)
    
    return processed_path

def ocr_image_with_tesseract(image_path):
    """Распознает текст с изображения через Tesseract (БЕСПЛАТНО!)"""
    try:
        # Предобработать изображение
        processed_path = preprocess_image(image_path)
        
        # Распознать текст
        text = pytesseract.image_to_string(processed_path, lang='rus')
        
        # Удалить обработанный файл
        os.remove(processed_path)
        
        print(f"✓ Текст распознан: {len(text)} символов")
        return text
    except Exception as e:
        print(f"✗ Ошибка OCR: {e}")
        return None

print("✓ OCR функции готовы (Tesseract)")

## 📋 БЛОК 7: LLM - Форматирование через Groq (БЕСПЛАТНО!)

In [ ]:
def format_menu_with_groq(raw_text, firm_id):
    """Отправляет текст в Groq (бесплатный LLM) для форматирования"""
    
    # Инициализируем Groq клиент
    client = Groq(api_key=GROQ_API_KEY)
    
    prompt = f"""Ты помощник по парсингу меню ресторана. Вот текст, распознанный с фотографии меню:

{raw_text}

Твоя задача:
1. Извлечь все товары (блюда, напитки) и их цены
2. Выформатировать в формат: firm_id | Товар | Цена
3. Убрать лишние символы, исправить опечатки OCR
4. Вернуть результат СНАЧАЛА как CSV (по одной линии за строку), ЗАТЕМ как XML

Используй firm_id: {firm_id}

Формат CSV для каждой строки:
{firm_id}|Название товара|Цена

После CSV выведи XML:
<?xml version="1.0" encoding="UTF-8"?>
<menu>
  <firm id="{firm_id}">
    <item>
      <n>Название товара</n>
      <price>Цена</price>
    </item>
  </firm>
</menu>

Будь точен и не добавляй лишних комментариев. CSV сначала, потом XML.
"""
    
    try:
        # Используем быструю модель Groq (бесплатно, 8000 токенов в минуту)
        response = client.chat.completions.create(
            model="mixtral-8x7b-32768",  # Бесплатная модель Groq
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=2000,
            temperature=0.3  # Низкая температура для точности
        )
        
        result = response.choices[0].message.content
        print(f"✓ Groq обработал текст (бесплатно!)")
        return result
        
    except Exception as e:
        print(f"✗ Ошибка Groq: {e}")
        return None

print("✓ Groq функция готова (БЕСПЛАТНАЯ LLM API)")

## 📋 БЛОК 8: Сохранение результатов

In [ ]:
def save_results(parsed_text, firm_id, folders):
    """Сохраняет результаты в CSV и XML файлы"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Извлекаем CSV и XML части
    csv_lines = []
    xml_content = ""
    
    lines = parsed_text.split('\n')
    in_xml = False
    
    for line in lines:
        if line.startswith('<?xml'):
            in_xml = True
        
        if in_xml:
            xml_content += line + '\n'
        elif '|' in line and not line.startswith('#') and firm_id in line:
            csv_lines.append(line.strip())
    
    # Сохраняем CSV
    csv_path = folders['results'] / f"menu_{firm_id}_{timestamp}.csv"
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write("firm_id|Товар|Цена\n")
        for line in csv_lines:
            f.write(line + '\n')
    
    print(f"✓ CSV сохранен: {csv_path.name}")
    
    # Сохраняем XML
    xml_path = folders['results'] / f"menu_{firm_id}_{timestamp}.xml"
    with open(xml_path, 'w', encoding='utf-8') as f:
        if xml_content:
            f.write(xml_content)
        else:
            # Если XML не распознан, создаем сами
            f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
            f.write('<menu>\n')
            f.write(f'  <firm id="{firm_id}">\n')
            for line in csv_lines:
                parts = line.split('|')
                if len(parts) >= 3:
                    f.write(f'    <item>\n')
                    f.write(f'      <n>{parts[1]}</n>\n')
                    f.write(f'      <price>{parts[2]}</price>\n')
                    f.write(f'    </item>\n')
            f.write('  </firm>\n')
            f.write('</menu>\n')
    
    print(f"✓ XML сохранен: {xml_path.name}")
    
    # Создаем лог обработки
    log_path = folders['processed'] / f"log_{firm_id}.json"
    log_data = {
        'firm_id': firm_id,
        'timestamp': datetime.now().isoformat(),
        'photos_processed': 1,
        'status': 'completed'
    }
    
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Лог обработки создан")
    
    return csv_path, xml_path

print("✓ Функции сохранения готовы")

## 🚀 БЛОК 9: ГЛАВНАЯ ФУНКЦИЯ - ЗАПУСК ПАРСИНГА

In [ ]:
def process_menu(firm_id=FIRM_ID, menu_image_paths=None):
    """Главная функция для обработки меню"""
    
    print(f"\n{'='*60}")
    print(f"ЗАПУСК ПАРСИНГА МЕНЮ")
    print(f"{'='*60}")
    print(f"📍 Firm ID: {firm_id}")
    print(f"⏰ Время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Если пути не переданы, ищем в папке photos
    if menu_image_paths is None:
        photos_dir = folders['photos']
        menu_image_paths = list(photos_dir.glob('*.jpg')) + list(photos_dir.glob('*.png'))
    
    if not menu_image_paths:
        print("✗ Изображения меню не найдены!")
        print(f"Загрузите фото в: {folders['photos']}")
        return
    
    print(f"📸 Найдено изображений: {len(menu_image_paths)}\n")
    
    all_recognized_text = ""
    
    # Обрабатываем каждое изображение
    for idx, image_path in enumerate(menu_image_paths, 1):
        print(f"\n--- Обработка {idx}/{len(menu_image_paths)}: {image_path.name} ---")
        
        # OCR с Tesseract (БЕСПЛАТНО!)
        recognized_text = ocr_image_with_tesseract(str(image_path))
        
        if recognized_text:
            all_recognized_text += f"\n--- Страница {idx} ---\n{recognized_text}\n"
        else:
            print(f"⚠️ Не удалось распознать текст из {image_path.name}")
    
    # Отправляем весь текст в Groq LLM для форматирования (БЕСПЛАТНО!)
    print(f"\n🤖 Отправка текста в Groq для форматирования...")
    formatted_result = format_menu_with_groq(all_recognized_text, firm_id)
    
    if not formatted_result:
        print("✗ Ошибка при обработке текста")
        return
    
    # Сохраняем результаты
    print(f"\n💾 Сохранение результатов...")
    csv_path, xml_path = save_results(formatted_result, firm_id, folders)
    
    print(f"\n{'='*60}")
    print(f"✓ ПАРСИНГ ЗАВЕРШЕН УСПЕШНО!")
    print(f"{'='*60}")
    print(f"📄 CSV: {csv_path.name}")
    print(f"📋 XML: {xml_path.name}")
    print(f"\nРезультаты сохранены в Google Drive")
    print(f"Путь: {folders['results']}")
    
    return formatted_result

print("✓ Главная функция готова")

## 📥 БЛОК 10: ЗАГРУЗКА ФОТ0 С ИНТЕРНЕТА (ОПЦИОНАЛЬНО)

Если хотите загрузить фото напрямую с 2ГИС, используйте этот код:

In [ ]:
# ПРИМЕР: Загрузить фото с URL 2ГИС
# (раскомментируйте и отредактируйте URL)

# urls = [
#     "https://i1.photo.2gis.com/photo-gallery/f5f27f8e-f12d-4495-8a2e-5bd98297bcf9_328x170.jpg",
#     "https://i7.photo.2gis.com/photo-gallery/d06fb30e-b678-4f87-981c-9e82a327e753_328x170.jpg",
#     "https://i0.photo.2gis.com/photo-gallery/17a98376-d6cc-4ed0-a6bf-233caf59350a_328x170.jpg",
#     "https://i0.photo.2gis.com/photo-gallery/d3deac79-bad2-49f8-9ad7-5cc6d781cfa3_328x170.jpg"
# ]

# print("📥 Загрузка фото с 2ГИС...\n")
# for i, url in enumerate(urls, 1):
#     print(f"Загрузка {i}/4...")
#     download_image(url, folders['photos'] / f"menu_page{i}.jpg")

print("✓ Блок загрузки готов (закомментирован)")

## ⚡ БЛОК 11: ЗАПУСТИТЕ ЭТО!

### Инструкция:
1. ✅ Отредактируйте BLOK 3 - вставьте Groq API ключ
2. ✅ Загрузите 1-4 фото меню в папку `photos` на Google Drive
3. ✅ Запустите ячейку ниже

In [ ]:
# 🚀 ОСНОВНАЯ КОМАНДА - ЗАПУСТИТЕ ЭТО!

process_menu(firm_id=FIRM_ID)

## 📊 БЛОК 12: ПРОВЕРКА РЕЗУЛЬТАТОВ

In [ ]:
# Посмотреть CSV результаты
import pandas as pd

csv_files = list(folders['results'].glob("menu_*.csv"))
if csv_files:
    latest_csv = max(csv_files, key=lambda p: p.stat().st_mtime)
    print(f"📄 Файл: {latest_csv.name}\n")
    df = pd.read_csv(latest_csv, delimiter='|')
    print(df.to_string(index=False))
else:
    print("✗ CSV файлы не найдены")

In [ ]:
# Посмотреть XML результаты

xml_files = list(folders['results'].glob("menu_*.xml"))
if xml_files:
    latest_xml = max(xml_files, key=lambda p: p.stat().st_mtime)
    print(f"📋 Файл: {latest_xml.name}\n")
    with open(latest_xml) as f:
        print(f.read())
else:
    print("✗ XML файлы не найдены")

## 🔄 БЛОК 13: ОБРАБОТАТЬ НЕСКОЛЬКО РЕСТОРАНОВ

Если нужно обработать разные заведения:

In [ ]:
# Пример: обработать несколько ресторанов

# restaurants = {
#     "70000001048209528": "Coffee Story",
#     "70000001234567890": "Другой ресторан",
#     "70000001111111111": "Еще один"
# }

# for firm_id, name in restaurants.items():
#     print(f"\n🏪 {name}")
#     # Убедитесь что фото для этого ресторана в папке photos
#     process_menu(firm_id=firm_id)

print("✓ Мультиресторан обработка готова (закомментирована)")

## 📝 ЧЕКЛИСТ:

- [ ] Установлены зависимости (БЛОК 1)
- [ ] Импорты загружены (БЛОК 2)
- [ ] Groq API ключ вставлен (БЛОК 3)
- [ ] Google Drive подключен (БЛОК 4)
- [ ] Фото загружены в папку `photos`
- [ ] Запущен БЛОК 11 (`process_menu()`)
- [ ] Результаты видны в БЛОКЕ 12

## 📍 Ссылки:

- 🔑 **Groq API (БЕСПЛАТНО)**: https://console.groq.com/
- 📁 **Google Drive**: https://drive.google.com/
- 📓 **Google Colab**: https://colab.research.google.com/

## 🎯 РЕЗУЛЬТАТЫ:

**Google Drive > Menu_Parser > results >**
- `menu_70000001048209528_ДАТА_ВРЕМЯ.csv` ← CSV результат
- `menu_70000001048209528_ДАТА_ВРЕМЯ.xml` ← XML результат